# Additional Sanity Checks Beyond the Paper Examples

This notebook adds implementation-level sanity checks that are not direct reproductions of the paper tables.

The goal is to verify that the SABR simulation package is:

1. reproducible under fixed random seeds;
2. consistent with the absorbing boundary at zero;
3. numerically stable under stressed but valid SABR parameter sets;
4. consistent with basic no-arbitrage option-price shape conditions.

These checks are meant to complement the paper reproduction notebook and the existing sanity-check notebook, without modifying prior work.

## 1. Fixed-seed reproducibility check

A simulation package should be reproducible when the same random seed is used.

This check runs the same SABR configuration twice with the same seed and once with a different seed.

Expected result:

- same seed: terminal samples should be exactly identical;
- different seed: terminal samples should change;
- same-seed prices and means should match exactly.

In [2]:
params = SABRParams(
    f0=1.0,
    sigma0=0.25,
    nu=0.45,
    rho=-0.4,
    beta=0.6,
)

mc_same_seed = MonteCarloConfig(
    maturity=2.0,
    step=0.25,
    n_paths=30_000,
    seed=20260426,
)

mc_different_seed = MonteCarloConfig(
    maturity=2.0,
    step=0.25,
    n_paths=30_000,
    seed=20260427,
)

terminal_a = simulate_terminal_forward(params, mc_same_seed)
terminal_b = simulate_terminal_forward(params, mc_same_seed)
terminal_c = simulate_terminal_forward(params, mc_different_seed)

seed_check = pd.DataFrame(
    [
        {
            "check": "same seed gives identical samples",
            "result": bool(np.array_equal(terminal_a, terminal_b)),
        },
        {
            "check": "different seed changes samples",
            "result": bool(not np.array_equal(terminal_a, terminal_c)),
        },
        {
            "check": "same seed mean A",
            "result": float(np.mean(terminal_a)),
        },
        {
            "check": "same seed mean B",
            "result": float(np.mean(terminal_b)),
        },
        {
            "check": "different seed mean",
            "result": float(np.mean(terminal_c)),
        },
        {
            "check": "same seed ATM call A",
            "result": european_call_price(terminal_a, params.f0),
        },
        {
            "check": "same seed ATM call B",
            "result": european_call_price(terminal_b, params.f0),
        },
        {
            "check": "different seed ATM call",
            "result": european_call_price(terminal_c, params.f0),
        },
    ]
)

seed_check

,check,result
0,same seed gives identical samples,True
1,different seed changes samples,True
2,same seed mean A,0.998195
3,same seed mean B,0.998195
4,different seed mean,0.999164
5,same seed ATM call A,0.139565
6,same seed ATM call B,0.139565
7,different seed ATM call,0.140169


## 2. Absorbing boundary check at zero

For \(0 < \beta < 1\), the SABR/CEV-style forward process has an absorbing boundary at zero.

This means that if the forward price starts at zero, it should remain zero.

This is not a paper table reproduction. It is a structural check of the simulation logic.

In [3]:
absorbing_params = SABRParams(
    f0=0.0,
    sigma0=0.25,
    nu=0.4,
    rho=-0.5,
    beta=0.6,
)

absorbing_rows = []

for maturity in [0.25, 1.0, 3.0]:
    mc = MonteCarloConfig(
        maturity=maturity,
        step=0.25,
        n_paths=20_000,
        seed=9000 + int(100 * maturity),
    )
    
    terminal = simulate_terminal_forward(absorbing_params, mc)
    
    absorbing_rows.append(
        {
            "maturity": maturity,
            "all_zero": bool(np.all(terminal == 0.0)),
            "all_finite": bool(np.isfinite(terminal).all()),
            "all_nonnegative": bool(np.all(terminal >= 0.0)),
            "mean_terminal": float(np.mean(terminal)),
            "max_terminal": float(np.max(terminal)),
        }
    )

absorbing_check = pd.DataFrame(absorbing_rows)
absorbing_check

,maturity,all_zero,all_finite,all_nonnegative,mean_terminal,max_terminal
0,0.250000,True,True,True,0.000000,0.000000
1,1.000000,True,True,True,0.000000,0.000000
2,3.000000,True,True,True,0.000000,0.000000


## 3. Stress-parameter stability check

This check uses stressed but valid SABR parameters.

The purpose is not to match a specific paper table. Instead, we want to confirm that the simulator remains stable when correlation, beta, and vol-of-vol are pushed toward difficult regimes.

For each case, we check:

- terminal values are finite;
- terminal values are nonnegative;
- the terminal forward mean is close to the initial forward within Monte Carlo noise;
- the ATM call price is finite and nonnegative.

In [8]:
stress_cases = [
    {
        "case": "high positive rho, low beta",
        "rho": 0.99,
        "beta": 0.20,
        "nu": 0.80,
    },
    {
        "case": "high negative rho, high beta",
        "rho": -0.99,
        "beta": 0.80,
        "nu": 0.80,
    },
    {
        "case": "zero rho, high vol-of-vol",
        "rho": 0.00,
        "beta": 0.30,
        "nu": 1.00,
    },
    {
        "case": "near lognormal beta",
        "rho": -0.60,
        "beta": 0.95,
        "nu": 0.50,
    },
]

stress_rows = []

for i, cfg in enumerate(stress_cases):
    params = SABRParams(
        f0=1.0,
        sigma0=0.25,
        nu=cfg["nu"],
        rho=cfg["rho"],
        beta=cfg["beta"],
    )
    
    mc = MonteCarloConfig(
        maturity=2.0,
        step=0.5,
        n_paths=25_000,
        seed=7000 + i,
    )
    
    terminal = simulate_terminal_forward(params, mc)
    
    mean_terminal = float(np.mean(terminal))
    std_terminal = float(np.std(terminal, ddof=1))
    stderr_terminal = std_terminal / math.sqrt(mc.n_paths)
    martingale_error = mean_terminal - params.f0
    z_score = martingale_error / stderr_terminal if stderr_terminal > 0 else np.nan
    atm_call = european_call_price(terminal, params.f0)
    
    stress_rows.append(
        {
            "case": cfg["case"],
            "rho": cfg["rho"],
            "beta": cfg["beta"],
            "nu": cfg["nu"],
            "all_finite": bool(np.isfinite(terminal).all()),
            "all_nonnegative": bool(np.all(terminal >= 0.0)),
            "mean_terminal": mean_terminal,
            "martingale_error": martingale_error,
            "stderr_terminal": stderr_terminal,
            "z_score": z_score,
            "abs_z_less_than_3": bool(abs(z_score) < 3.0),
            "atm_call_price": atm_call,
            "atm_call_nonnegative": bool(atm_call >= 0.0),
        }
    )

stress_check = pd.DataFrame(stress_rows)
stress_check[
    [
        "case",
        "rho",
        "beta",
        "nu",
        "all_finite",
        "all_nonnegative",
        "mean_terminal",
        "martingale_error",
        "z_score",
        "abs_z_less_than_3",
        "atm_call_price",
    ]
]

,case,rho,beta,nu,all_finite,all_nonnegative,mean_terminal,martingale_error,z_score,abs_z_less_than_3,atm_call_price
0,"high positive rho, low beta",0.990000,0.200000,0.800000,True,True,0.997357,-0.002643,-0.542780,True,0.139949
1,"high negative rho, high beta",-0.990000,0.800000,0.800000,True,True,0.999576,-0.000424,-0.211396,True,0.122284
2,"zero rho, high vol-of-vol",0.000000,0.300000,1.000000,True,True,0.995565,-0.004435,-1.355134,True,0.149451
3,near lognormal beta,-0.600000,0.950000,0.500000,True,True,1.000795,0.000795,0.354572,True,0.137233


## 4. No-arbitrage strike-shape check for European calls

European call prices should satisfy basic no-arbitrage shape conditions as a function of strike.

Using the same terminal sample for all strikes, we check:

1. call prices decrease as strike increases;
2. call prices are convex in strike;
3. call prices stay within simple lower and upper bounds.

This is a useful sanity check because it tests the simulated terminal distribution through option prices, rather than only checking terminal means.

In [5]:
shape_params = SABRParams(
    f0=1.0,
    sigma0=0.25,
    nu=0.5,
    rho=-0.55,
    beta=0.5,
)

shape_mc = MonteCarloConfig(
    maturity=3.0,
    step=0.5,
    n_paths=50_000,
    seed=424242,
)

terminal = simulate_terminal_forward(shape_params, shape_mc)

# Use an equally spaced strike grid so that discrete convexity is easy to check.
strikes = np.linspace(0.5, 1.5, 11)
call_prices = np.array([european_call_price(terminal, k) for k in strikes])

# Basic finite-difference checks.
first_diff = np.diff(call_prices)
second_diff = call_prices[:-2] - 2.0 * call_prices[1:-1] + call_prices[2:]

mean_forward = float(np.mean(terminal))
lower_bounds = np.maximum(mean_forward - strikes, 0.0)
upper_bounds = np.full_like(strikes, mean_forward)

tol = 1e-10

shape_table = pd.DataFrame(
    {
        "strike": strikes,
        "call_price": call_prices,
        "lower_bound_max_meanF_minus_K_0": lower_bounds,
        "upper_bound_meanF": upper_bounds,
        "above_lower_bound": call_prices + tol >= lower_bounds,
        "below_upper_bound": call_prices <= upper_bounds + tol,
    }
)

shape_summary = pd.DataFrame(
    [
        {
            "check": "call prices decrease with strike",
            "result": bool(np.all(first_diff <= tol)),
            "worst_first_difference": float(np.max(first_diff)),
        },
        {
            "check": "call prices are convex in strike",
            "result": bool(np.all(second_diff >= -tol)),
            "worst_second_difference": float(np.min(second_diff)),
        },
        {
            "check": "call prices above lower bound",
            "result": bool(np.all(call_prices + tol >= lower_bounds)),
            "worst_lower_bound_gap": float(np.min(call_prices - lower_bounds)),
        },
        {
            "check": "call prices below upper bound",
            "result": bool(np.all(call_prices <= upper_bounds + tol)),
            "worst_upper_bound_gap": float(np.max(call_prices - upper_bounds)),
        },
        {
            "check": "all terminal values finite",
            "result": bool(np.isfinite(terminal).all()),
            "worst_first_difference": np.nan,
        },
        {
            "check": "all terminal values nonnegative",
            "result": bool(np.all(terminal >= 0.0)),
            "worst_first_difference": np.nan,
        },
    ]
)

shape_summary

,check,result,worst_first_difference,worst_second_difference,worst_lower_bound_gap,worst_upper_bound_gap
0,call prices decrease with strike,True,-0.012520,NaN,NaN,NaN
1,call prices are convex in strike,True,NaN,0.004562,NaN,NaN
2,call prices above lower bound,True,NaN,NaN,0.024789,NaN
3,call prices below upper bound,True,NaN,NaN,NaN,-0.463983
4,all terminal values finite,True,NaN,NaN,NaN,NaN
5,all terminal values nonnegative,True,NaN,NaN,NaN,NaN


In [6]:
shape_table

,strike,call_price,lower_bound_max_meanF_minus_K_0,upper_bound_meanF,above_lower_bound,below_upper_bound
0,0.500000,0.534257,0.498240,0.998240,True,True
1,0.600000,0.449412,0.398240,0.998240,True,True
2,0.700000,0.369129,0.298240,0.998240,True,True
3,0.800000,0.294663,0.198240,0.998240,True,True
4,0.900000,0.227562,0.098240,0.998240,True,True
5,1.000000,0.169276,0.000000,0.998240,True,True
6,1.100000,0.121220,0.000000,0.998240,True,True
7,1.200000,0.083743,0.000000,0.998240,True,True
8,1.300000,0.056315,0.000000,0.998240,True,True
9,1.400000,0.037310,0.000000,0.998240,True,True


## 5. Compact summary

This final table collects the pass/fail style results from the additional sanity checks.

In [7]:
summary_rows = []

summary_rows.append(
    {
        "section": "fixed-seed reproducibility",
        "main_check": "same seed gives identical samples",
        "passed": bool(np.array_equal(terminal_a, terminal_b)),
    }
)

summary_rows.append(
    {
        "section": "fixed-seed reproducibility",
        "main_check": "different seed changes samples",
        "passed": bool(not np.array_equal(terminal_a, terminal_c)),
    }
)

summary_rows.append(
    {
        "section": "absorbing boundary",
        "main_check": "F0 = 0 remains absorbed at zero",
        "passed": bool(absorbing_check["all_zero"].all()),
    }
)

summary_rows.append(
    {
        "section": "absorbing boundary",
        "main_check": "absorbing-boundary samples are finite and nonnegative",
        "passed": bool(absorbing_check["all_finite"].all() and absorbing_check["all_nonnegative"].all()),
    }
)

summary_rows.append(
    {
        "section": "stress parameters",
        "main_check": "all stressed samples are finite",
        "passed": bool(stress_check["all_finite"].all()),
    }
)

summary_rows.append(
    {
        "section": "stress parameters",
        "main_check": "all stressed samples are nonnegative",
        "passed": bool(stress_check["all_nonnegative"].all()),
    }
)

summary_rows.append(
    {
        "section": "stress parameters",
        "main_check": "martingale z-scores are within 3 standard errors",
        "passed": bool(stress_check["abs_z_less_than_3"].all()),
    }
)

summary_rows.append(
    {
        "section": "no-arbitrage strike shape",
        "main_check": "call prices decrease with strike",
        "passed": bool(np.all(first_diff <= tol)),
    }
)

summary_rows.append(
    {
        "section": "no-arbitrage strike shape",
        "main_check": "call prices are convex in strike",
        "passed": bool(np.all(second_diff >= -tol)),
    }
)

summary_rows.append(
    {
        "section": "no-arbitrage strike shape",
        "main_check": "call prices satisfy simple bounds",
        "passed": bool(np.all(call_prices + tol >= lower_bounds) and np.all(call_prices <= upper_bounds + tol)),
    }
)

summary_df = pd.DataFrame(summary_rows)
summary_df

,section,main_check,passed
0,fixed-seed reproducibility,same seed gives identical samples,True
1,fixed-seed reproducibility,different seed changes samples,True
2,absorbing boundary,F0 = 0 remains absorbed at zero,True
3,absorbing boundary,absorbing-boundary samples are finite and nonn...,True
4,stress parameters,all stressed samples are finite,True
5,stress parameters,all stressed samples are nonnegative,True
6,stress parameters,martingale z-scores are within 3 standard errors,True
7,no-arbitrage strike shape,call prices decrease with strike,True
8,no-arbitrage strike shape,call prices are convex in strike,True
9,no-arbitrage strike shape,call prices satisfy simple bounds,True
